In [12]:
# ATIVIDADE 1: CHATBOT VERSÃO 1 (KNN)
# ==============================================================================
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# 1. Carregar dataset do CSV
df = pd.read_csv('dataset_moveis_100.csv')

# 2. Divisão Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(
    df['texto'], df['intencao'], test_size=0.30, random_state=42, stratify=df['intencao']
)

# TODO 1: Monte a Pipeline utilizando TfidfVectorizer e KNeighborsClassifier(n_neighbors=3, metric='cosine')
pipeline_knn = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', KNeighborsClassifier(n_neighbors=3, metric='cosine'))
])

# TODO 2: Treine a pipeline com os dados de treino (X_train, y_train)
pipeline_knn.fit(X_train, y_train)

# TODO 3: Gere as predicoes nos dados de teste e exiba o classification_report e a confusion_matrix
y_pred = pipeline_knn.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

LIMIAR_CONFIANCA = 0.50
print("\n=== INICIANDO BATERIA DE TESTES (10 INPUTS OBRIGATÓRIOS) ===")
for i in range(1, 11):
    print(f"\n[Teste {i}/10]")

    # TODO 4: Solicite a frase do usuario via teclado
    frase = input("Digite a frase do cliente: ").strip()

    # TODO 5: Extraia as probabilidades e a classe prevista usando predict_proba e predict
    probs = pipeline_knn.predict_proba([frase])[0]
    maior_prob = np.max(probs)
    intencao = pipeline_knn.predict([frase])[0]

    # TODO 6: Aplique a regra de decisao:
    # Se maior_prob >= LIMIAR_CONFIANCA: imprima a intencao e a probabilidade.
    # Senao: imprima o Fallback encaminhando para atendimento humano.
    if maior_prob >= LIMIAR_CONFIANCA:
        print(f"Intenção identificada: {intencao} (confiança: {maior_prob:.2%})")
    else:
        print(f"Confiança insuficiente ({maior_prob:.2%}). "
              f"Encaminhando para atendimento humano.")

                    precision    recall  f1-score   support

logistica_entregas       1.00      1.00      1.00        30
       reclamacoes       0.97      1.00      0.98        31
           suporte       1.00      0.97      0.98        30
 trocas_devolucoes       0.97      1.00      0.98        31
            vendas       1.00      0.97      0.98        31

          accuracy                           0.99       153
         macro avg       0.99      0.99      0.99       153
      weighted avg       0.99      0.99      0.99       153

[[30  0  0  0  0]
 [ 0 31  0  0  0]
 [ 0  1 29  0  0]
 [ 0  0  0 31  0]
 [ 0  0  0  1 30]]

=== INICIANDO BATERIA DE TESTES (10 INPUTS OBRIGATÓRIOS) ===

[Teste 1/10]


KeyboardInterrupt: Interrupted by user

In [13]:
# ATIVIDADE 2: CHATBOT VERSÃO 2 (DECISION TREE)
# ==============================================================================
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# 1. Carregar dataset do CSV
df = pd.read_csv('dataset_moveis_100.csv')

# 2. Divisão Treino e Teste (30% teste, estratificada)
X_train, X_test, y_train, y_test = train_test_split(
    df['texto'], df['intencao'], test_size=0.30, random_state=42, stratify=df['intencao']
)

# 3. Pipeline: TF-IDF + DecisionTreeClassifier
pipeline_tree = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

# 4. Treinamento
pipeline_tree.fit(X_train, y_train)

# 5. Avaliação: Matriz de Confusão e Relatório de Classificação
y_pred = pipeline_tree.predict(X_test)

print("=== Matriz de Confusão ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== Relatório de Classificação ===")
print(classification_report(y_test, y_pred))

# 6. Bateria de 8 testes interativos com fallback
LIMIAR_CONFIANCA = 0.50

print("\n=== INICIANDO BATERIA DE TESTES (8 INPUTS OBRIGATÓRIOS) ===")
for i in range(1, 9):
    print(f"\n[Teste {i}/8]")

    frase = input("Digite a frase do cliente: ").strip()

    probs = pipeline_tree.predict_proba([frase])[0]
    maior_prob = np.max(probs)
    intencao = pipeline_tree.predict([frase])[0]

    if maior_prob >= LIMIAR_CONFIANCA:
        print(f"Intenção identificada: {intencao} (confiança: {maior_prob:.2%})")
    else:
        print("Desculpe, não entendi sua solicitação. Encaminhando você para um atendente humano...")

=== Matriz de Confusão ===
[[30  0  0  0  0]
 [ 0 31  0  0  0]
 [ 0  1 29  0  0]
 [ 0  0  0 31  0]
 [ 3  0  0  0 28]]

=== Relatório de Classificação ===
                    precision    recall  f1-score   support

logistica_entregas       0.91      1.00      0.95        30
       reclamacoes       0.97      1.00      0.98        31
           suporte       1.00      0.97      0.98        30
 trocas_devolucoes       1.00      1.00      1.00        31
            vendas       1.00      0.90      0.95        31

          accuracy                           0.97       153
         macro avg       0.98      0.97      0.97       153
      weighted avg       0.98      0.97      0.97       153


=== INICIANDO BATERIA DE TESTES (8 INPUTS OBRIGATÓRIOS) ===

[Teste 1/8]


KeyboardInterrupt: Interrupted by user

In [14]:
# ==============================================================================
# ATIVIDADE 3: GERAÇÃO DO RELATÓRIO COMPARATIVO
# ==============================================================================
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# --- Métricas do KNN (reaproveitando y_test / y_pred da Atividade 1) ---
y_pred_knn = pipeline_knn.predict(X_test)
acc_knn = accuracy_score(y_test, y_pred_knn)
report_knn = classification_report(y_test, y_pred_knn, output_dict=True)
f1_knn = report_knn['weighted avg']['f1-score']
matriz_knn = confusion_matrix(y_test, y_pred_knn)

# --- Métricas da Decision Tree (reaproveitando y_test / y_pred da Atividade 2) ---
y_pred_tree = pipeline_tree.predict(X_test)
acc_tree = accuracy_score(y_test, y_pred_tree)
report_tree = classification_report(y_test, y_pred_tree, output_dict=True)
f1_tree = report_tree['weighted avg']['f1-score']
matriz_tree = confusion_matrix(y_test, y_pred_tree)

print("KNN -> Acurácia:", round(acc_knn * 100, 2), "% | F1 (weighted):", round(f1_knn * 100, 2), "%")
print("Matriz KNN:\n", matriz_knn)
print("\nDecision Tree -> Acurácia:", round(acc_tree * 100, 2), "% | F1 (weighted):", round(f1_tree * 100, 2), "%")
print("Matriz Tree:\n", matriz_tree)
print("\nClasses (ordem da matriz):", list(pipeline_knn.classes_))

# --- Geração do arquivo .md com os números já preenchidos ---
conteudo_md = f"""# Relatório de Avaliação NLU - SAC Móveis Residenciais

## 1. Tabela Comparativa de Métricas (Dados de Teste)

| Modelo | Acurácia Geral | F1-Score (Weighted) | Principais Erros na Matriz |
| :--- | :--- | :--- | :--- |
| **KNN (K=3)** | {acc_knn*100:.1f}% | {f1_knn*100:.1f}% | [PREENCHER: veja matriz_knn acima e descreva quais classes trocaram] |
| **Decision Tree** | {acc_tree*100:.1f}% | {f1_tree*100:.1f}% | [PREENCHER: veja matriz_tree acima e descreva quais classes trocaram] |

## 2. Análise dos Testes de Entrada (`input()`)

- **Comportamento do KNN (10 testes):** [PREENCHER: quantos dos seus 10 testes caíram no fallback? A confiança variou entre 0.33/0.67/1.0?]
- **Comportamento da Decision Tree (8 testes):** [PREENCHER: quantos dos seus 8 testes caíram no fallback? A confiança ficou perto de 100% na maioria?]

## 3. Veredito Final

- **Melhor modelo para este projeto:** [PREENCHER: KNN ou Decision Tree]
- **Justificativa técnica:** [PREENCHER: baseie-se nas métricas acima e no comportamento do fallback]
"""

with open('resultados_aula05.md', 'w', encoding='utf-8') as f:
    f.write(conteudo_md)

print("\n Arquivo 'resultados_aula05.md' gerado com as métricas preenchidas!")

KNN -> Acurácia: 98.69 % | F1 (weighted): 98.69 %
Matriz KNN:
 [[30  0  0  0  0]
 [ 0 31  0  0  0]
 [ 0  1 29  0  0]
 [ 0  0  0 31  0]
 [ 0  0  0  1 30]]

Decision Tree -> Acurácia: 97.39 % | F1 (weighted): 97.38 %
Matriz Tree:
 [[30  0  0  0  0]
 [ 0 31  0  0  0]
 [ 0  1 29  0  0]
 [ 0  0  0 31  0]
 [ 3  0  0  0 28]]

Classes (ordem da matriz): ['logistica_entregas', 'reclamacoes', 'suporte', 'trocas_devolucoes', 'vendas']

 Arquivo 'resultados_aula05.md' gerado com as métricas preenchidas!
